# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.5 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID='task214'
CH=10
H=W=30
TASK_PATH=Path(COMPETITION)/f'{TASK_ID}.json'

OUT_DIR=Path('.')
ONNX_PATH=OUT_DIR/f'{TASK_ID}.onnx'

In [6]:
data=json.load(open(TASK_PATH))
print({k:len(v) for k,v in data.items()})

{'train': 3, 'test': 1, 'arc-gen': 262}


In [7]:
def pad_grid(g,h=H,w=W):
    arr=np.zeros((h,w),dtype=np.int64)
    a=np.array(g,dtype=np.int64)
    arr[:a.shape[0],:a.shape[1]]=a
    return arr

def onehot_grid(g):
    arr=pad_grid(g)
    x=np.zeros((1,CH,H,W),dtype=np.float32)
    for c in range(CH):
        x[0,c]=(arr==c)
    return x

def pred_to_grid(y):
    if y.ndim==4: y=y[0]
    return y.argmax(axis=0).astype(np.int64)

class Task214RotFill(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer('zeros_tail', torch.zeros(1,CH,3,19))
        self.register_buffer('zeros_bottom', torch.zeros(1,CH,27,30))
    def forward(self,x):
        A=x[:,:,0:3,0:3]
        sep1=x[:,:,0:3,3:4]
        sep2=x[:,:,0:3,7:8]
        B=torch.flip(torch.transpose(A,2,3), dims=[3])
        C=torch.flip(torch.flip(A,dims=[2]), dims=[3])
        top=torch.cat([A,sep1,B,sep2,C,self.zeros_tail], dim=3)
        return torch.cat([top,self.zeros_bottom], dim=2)

model=Task214RotFill().eval()

In [8]:
# Export static ONNX
dummy=torch.zeros(1,CH,H,W,dtype=torch.float32)
torch.onnx.export(model,dummy,ONNX_PATH.as_posix(),input_names=['input'],output_names=['output'],opset_version=17,do_constant_folding=True,dynamic_axes=None,dynamo=False)
m=onnx.load(ONNX_PATH.as_posix())
onnx.checker.check_model(m)
m=shape_inference.infer_shapes(m)
onnx.save(m,ONNX_PATH.as_posix())
print('saved', ONNX_PATH, 'bytes', ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/2378069685.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,ONNX_PATH.as_posix(),input_names=['input'],output_names=['output'],opset_version=17,do_constant_folding=True,dynamic_axes=None,dynamo=False)


saved task214.onnx bytes 39739


In [9]:
# Raw padded ONNX validation: compare full 30x30 output, not cropped output.
sess=ort.InferenceSession(ONNX_PATH.as_posix(), providers=['CPUExecutionProvider'])
def check_split(split):
    ok=0
    for ex in data[split]:
        y=sess.run(None, {'input':onehot_grid(ex['input'])})[0]
        ok += int(np.array_equal(pred_to_grid(y), pad_grid(ex['output'])))
    return ok, len(data[split])
for split in ['train','test','arc-gen']:
    print(split, check_split(split))

train (3, 3)
test (1, 1)
arc-gen (262, 262)


In [10]:
# ONNX structural checks
m=onnx.load(ONNX_PATH.as_posix())
ops=sorted(set(n.op_type for n in m.graph.node))
forbidden={'Loop','Scan','NonZero','Unique','Script','Function'}
print('ops',ops)
print('forbidden', sorted(set(ops)&forbidden))
print('empty optional inputs', [(n.op_type, list(n.input)) for n in m.graph.node if any(i=='' for i in n.input)])
assert not (set(ops)&forbidden)
assert ONNX_PATH.stat().st_size < 1440000

ops ['Concat', 'Constant', 'Slice', 'Transpose']
forbidden []
empty optional inputs []


In [11]:
with zipfile.ZipFile('submission.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, 'task214.onnx')
print('wrote submission.zip')

wrote submission.zip
